# Customer Churn Prediction — Exploratory Data Analysis

**Project:** AICTE-IBM Machine Learning Internship  
**Dataset:** IBM Telco Customer Churn  
**Goal:** Understand patterns in customer data that predict churn (whether a customer cancels their telecom subscription).

---

## 1. Setup and Data Loading

In [ ]:
import os
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_theme(style='whitegrid', palette='muted')

# Add src/ to path so we can import preprocess.py
sys.path.insert(0, '../src')
from preprocess import load_and_clean, NUMERIC_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES

DATA_PATH = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
print('Libraries loaded successfully.')

In [ ]:
# Load raw CSV first for inspection BEFORE cleaning
df_raw = pd.read_csv(DATA_PATH)
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 2. Dataset Overview

In [ ]:
print('=== Data Types and Non-Null Counts ===')
df_raw.info()

In [ ]:
print('=== Basic Statistics (Numeric Columns) ===')
df_raw.describe()

In [ ]:
# Check for duplicate rows
print(f'Duplicate rows: {df_raw.duplicated().sum()}')

# Check TotalCharges — it's object dtype but should be numeric
print(f"\nTotalCharges dtype: {df_raw['TotalCharges'].dtype}")
print(f"Non-numeric TotalCharges values: {pd.to_numeric(df_raw['TotalCharges'], errors='coerce').isna().sum()}")
print("\nSample of problematic TotalCharges rows:")
problem_rows = df_raw[pd.to_numeric(df_raw['TotalCharges'], errors='coerce').isna()]
print(problem_rows[['customerID', 'tenure', 'TotalCharges', 'Churn']].head(5))

### Finding
The `TotalCharges` column is stored as text (object dtype). About **11 rows** contain only whitespace (`" "`) instead of a number. These are new customers with `tenure = 0` (no charges accumulated yet). We drop these rows since they represent an insignificant fraction of 7,043 rows.

In [ ]:
# Load the cleaned version for all further analysis
df = load_and_clean(DATA_PATH)
print(f'\nCleaned dataset shape: {df.shape}')
df.head()

## 3. Churn Distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print('Churn Value Counts:')
print(pd.DataFrame({'Count': churn_counts, 'Percentage': churn_pct.round(1)}))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
labels = ['No Churn (0)', 'Churn (1)']
axes[0].bar(labels, churn_counts.values, color=['#4CAF50', '#F44336'], edgecolor='white', width=0.5)
axes[0].set_title('Churn Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=11)

# Pie chart
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'],
            autopct='%1.1f%%', colors=['#4CAF50', '#F44336'], startangle=90)
axes[1].set_title('Churn Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Customer Churn Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../models/eda_churn_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

### Finding
The dataset is **imbalanced**: about **26.6% of customers churned** and 73.4% stayed. This means accuracy alone is not a good metric — a model that predicts "No Churn" for everyone would get ~73% accuracy without being useful. We use **ROC-AUC and F1-score** as primary metrics.

## 4. Churn by Contract Type

In [ ]:
# Re-create churn label for plotting
df['Churn_Label'] = df['Churn'].map({1: 'Churn', 0: 'No Churn'})

# Need original Contract from raw — already in df since we kept all columns
fig, ax = plt.subplots(figsize=(8, 5))
contract_churn = df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack() * 100
contract_churn.rename(columns={0: 'No Churn', 1: 'Churn'}, inplace=True)
contract_churn.plot(kind='bar', ax=ax, color=['#4CAF50', '#F44336'],
                    edgecolor='white', width=0.6)
ax.set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Contract Type', fontsize=11)
ax.set_ylabel('Percentage of Customers (%)', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Churn')
ax.yaxis.grid(True, alpha=0.5)
plt.tight_layout()
plt.savefig('../models/eda_churn_by_contract.png', bbox_inches='tight', dpi=120)
plt.show()

### Finding
**Month-to-month** contract customers have a dramatically higher churn rate (~43%) compared to **one-year** (~11%) and **two-year** contracts (~3%). Contract type is one of the **strongest predictors** of churn. Long-term contract customers are much more likely to stay.

## 5. Churn by Tenure

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
df[df['Churn'] == 0]['tenure'].hist(bins=30, alpha=0.7, color='#4CAF50',
                                     label='No Churn', ax=ax, edgecolor='white')
df[df['Churn'] == 1]['tenure'].hist(bins=30, alpha=0.7, color='#F44336',
                                     label='Churn', ax=ax, edgecolor='white')
ax.set_title('Customer Tenure Distribution by Churn', fontsize=13, fontweight='bold')
ax.set_xlabel('Tenure (months)', fontsize=11)
ax.set_ylabel('Number of Customers', fontsize=11)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../models/eda_churn_by_tenure.png', bbox_inches='tight', dpi=120)
plt.show()

print(f"Average tenure — No Churn: {df[df['Churn']==0]['tenure'].mean():.1f} months")
print(f"Average tenure — Churn:    {df[df['Churn']==1]['tenure'].mean():.1f} months")

### Finding
**New customers (low tenure) churn the most.** Customers who have been with the company for 1–12 months are far more likely to leave. Long-tenured customers (50+ months) almost never churn. This suggests **early engagement and retention programs** could significantly reduce churn.

## 6. Churn by Monthly Charges

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
df[df['Churn'] == 0]['MonthlyCharges'].hist(bins=30, alpha=0.7, color='#4CAF50',
                                             label='No Churn', ax=ax, edgecolor='white')
df[df['Churn'] == 1]['MonthlyCharges'].hist(bins=30, alpha=0.7, color='#F44336',
                                             label='Churn', ax=ax, edgecolor='white')
ax.set_title('Monthly Charges Distribution by Churn', fontsize=13, fontweight='bold')
ax.set_xlabel('Monthly Charges ($)', fontsize=11)
ax.set_ylabel('Number of Customers', fontsize=11)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../models/eda_churn_by_monthly_charges.png', bbox_inches='tight', dpi=120)
plt.show()

print(f"Average monthly charges — No Churn: ${df[df['Churn']==0]['MonthlyCharges'].mean():.2f}")
print(f"Average monthly charges — Churn:    ${df[df['Churn']==1]['MonthlyCharges'].mean():.2f}")

### Finding
Churned customers tend to pay **higher monthly charges** on average (~$74) compared to customers who stayed (~$61). This may indicate that customers feel they are not getting value for higher-cost plans, or are attracted away by cheaper competitors.

## 7. Churn by Internet Service

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
internet_churn = df.groupby('InternetService')['Churn'].value_counts(normalize=True).unstack() * 100
internet_churn.rename(columns={0: 'No Churn', 1: 'Churn'}, inplace=True)
internet_churn.plot(kind='bar', ax=ax, color=['#4CAF50', '#F44336'],
                    edgecolor='white', width=0.6)
ax.set_title('Churn Rate by Internet Service Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Internet Service Type', fontsize=11)
ax.set_ylabel('Percentage (%)', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Churn')
ax.yaxis.grid(True, alpha=0.5)
plt.tight_layout()
plt.savefig('../models/eda_churn_by_internet.png', bbox_inches='tight', dpi=120)
plt.show()

### Finding
**Fiber optic** internet service customers churn at a much higher rate (~42%) compared to **DSL** (~19%) or customers with **no internet service** (~7%). Fiber optic is a premium, higher-cost service — possibly attracting price-sensitive customers who switch when a better deal appears.

## 8. Churn by Payment Method

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
payment_churn = df.groupby('PaymentMethod')['Churn'].value_counts(normalize=True).unstack() * 100
payment_churn.rename(columns={0: 'No Churn', 1: 'Churn'}, inplace=True)
payment_churn.plot(kind='bar', ax=ax, color=['#4CAF50', '#F44336'],
                    edgecolor='white', width=0.6)
ax.set_title('Churn Rate by Payment Method', fontsize=13, fontweight='bold')
ax.set_xlabel('Payment Method', fontsize=11)
ax.set_ylabel('Percentage (%)', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.legend(title='Churn')
ax.yaxis.grid(True, alpha=0.5)
plt.tight_layout()
plt.savefig('../models/eda_churn_by_payment.png', bbox_inches='tight', dpi=120)
plt.show()

### Finding
Customers paying via **electronic check** have the highest churn rate (~45%). Customers using **automatic payment methods** (bank transfer, credit card — both auto) churn at much lower rates (~15–17%). Automatic payments suggest higher engagement and commitment.

## 9. Correlation Heatmap (Numeric Features)

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))  # show lower triangle only
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/eda_correlation_heatmap.png', bbox_inches='tight', dpi=120)
plt.show()

### Finding
- **Tenure and Churn** are negatively correlated (−0.35): longer-tenured customers churn less.
- **MonthlyCharges and Churn** are positively correlated (+0.19): higher charges → more churn.
- **Tenure and TotalCharges** are strongly correlated (+0.83): naturally, longer tenure accumulates more charges. We should be careful of multicollinearity, but since we are using tree-based and regularised linear models, this is manageable.

## 10. Summary of Key Findings

| Feature | Observation |
|---------|-------------|
| **Contract Type** | Month-to-month customers churn ~43% vs. 3% for two-year contracts |
| **Tenure** | Short-tenured customers (< 12 months) are high churn risk |
| **Monthly Charges** | Higher charges correlate with higher churn |
| **Internet Service** | Fiber optic customers churn at ~42% |
| **Payment Method** | Electronic check users churn ~45% |
| **Class Imbalance** | ~26.6% churn rate — use balanced class weights & ROC-AUC metric |

These insights guide both the feature selection and the model interpretation.